In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


In [17]:
# Charger les données train et test à partir des fichiers CSV
train=pd.read_csv("../data/train.csv")
test=pd.read_csv("../data/test.csv")
print("LA dimension du train est:",train.shape)
print("LA dimension du test est:",test.shape)

LA dimension du train est: (76518, 38)
LA dimension du test est: (51012, 37)


In [18]:
class_count =train['Target'].value_counts(normalize=True)*100
print("La proportion d'etudiants pour chaque classe est:")
print(class_count)


La proportion d'etudiants pour chaque classe est:
Target
Graduate    47.416294
Dropout     33.058888
Enrolled    19.524818
Name: proportion, dtype: float64


In [19]:
#Separation des variables explicatives et de la variable cible
X=train.drop(columns=['id','Target'])
y=train['Target']
print("La dimension de X est:",X.shape)
print("La dimension de y est:",y.shape)

La dimension de X est: (76518, 36)
La dimension de y est: (76518,)


In [20]:
#Utiliser toutes les données étiquées
X_dev=X.copy()
y_dev=y.copy()
#Preparer les variables explicatives du fichier test
X_test=test.drop(columns=['id']).copy()

In [21]:
# feature engineering pour la creation de nouvelles variables dans notre data frame
def feature_engineering(df): 
    df_fe=df.copy()
    # 1. le taux de modules validés pour le 1er et 2eme  semestre
    df_fe['approval_rate_sem1']=df_fe['Curricular units 1st sem (approved)']/(df_fe['Curricular units 1st sem (enrolled)']+1e-5)
    
    df_fe['approval_rate_sem2']=df_fe['Curricular units 2nd sem (approved)']/( df_fe['Curricular units 2nd sem (enrolled)']+1e-5)
    
    # 2. le nombre  de modules validés pour l'ensemble des deux semestres
    df_fe['total_approved']=df_fe['Curricular units 1st sem (approved)']+df_fe['Curricular units 2nd sem (approved)']
    
    #3.le nombre de modules suivi pour l'ensemble des deux semestres
    df_fe['total_enrolled']=df_fe['Curricular units 1st sem (enrolled)']+df_fe['Curricular units 2nd sem (enrolled)']
    
    #4. le taux de reussite pour l'ensemble des deux semestres
    df_fe['overall_approval_rate']=df_fe['total_approved']/(df_fe['total_enrolled']+ 1e-5)
    
    # 5. La progression de la note entre le 1er et le 2eme semestre
    df_fe['grade_progression']=df_fe['Curricular units 2nd sem (grade)']-df_fe['Curricular units 1st sem (grade)']
    
    # 6. Progression du nombres de  modules validés entre les deux semestres
    df_fe["approved_progression"]=df_fe["Curricular units 2nd sem (approved)"]-df_fe["Curricular units 1st sem (approved)"] 
    # 7.Instabilité Financière
    df_fe['financial_instability'] = (df_fe['Debtor'] == 1).astype(int) - (df_fe['Tuition fees up to date'] == 1).astype(int) - (df_fe['Scholarship holder'] == 1).astype(int)
   

    return  df_fe 
            
X_dev_fe=feature_engineering(X_dev)
X_test_fe=feature_engineering(X_test)
print("La dimension de X_dev_fe est:",X_dev_fe.shape)
print("La dimension de X_test_fe est:",X_test_fe.shape)

La dimension de X_dev_fe est: (76518, 44)
La dimension de X_test_fe est: (51012, 44)


In [ ]:
# definir les variables catégorielles
cat_cols=['Marital status', 'Application mode', 'Course', 'Previous qualification', 
    'Nacionality', "Mother's qualification", "Father's qualification", 
    "Mother's occupation", "Father's occupation",'Educational special needs','Displaced',
    'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'International']  

In [23]:
# Preparer  les variables catégorielles pour  Lightbm et Xgboost 
X_dev_model= X_dev_fe.copy()
X_test_model=X_test_fe.copy()

#Harmoniser les variables 
for col in cat_cols:
  categories=pd.concat([ X_dev_model[col],  X_test_model[col]]).dropna().unique()
  X_dev_model[col]= pd.Categorical(X_dev_model[col],categories=categories)
  X_test_model[col]= pd.Categorical(X_test_model[col],categories=categories)
    

# Preparer les données pour Catboost
X_dev_cb=X_dev_model.copy()
X_test_cb=X_test_model.copy()
       
#Encoder la variable cible Target 
encoder=LabelEncoder()
y_dev_encoded=encoder.fit_transform(y_dev)

In [36]:


#Validation croisée sur 5 plis pour évaluer les performances du  modèle XGBoost et de LightGBM  
skf= StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

scores_xgb=[]
scores_lgbm=[]

oof_xgb=np.zeros((len(X_dev_model), len(encoder.classes_)))
oof_lgbm=np.zeros((len(X_dev_model), len(encoder.classes_)))

# Les probabilités predites sur les données test
test_probs_xgb=np.zeros((len(X_test_model),len(encoder.classes_)))
test_probs_lgbm=np.zeros((len(X_test_model),len(encoder.classes_)))

for fold,(train_index, val_index) in enumerate(skf.split(X_dev_model,y_dev_encoded)):
    print(f"----Fold {fold+1}---")
    X_train_fold= X_dev_model.iloc[train_index]
    y_train_fold= y_dev_encoded[train_index]
   
    X_val_fold= X_dev_model.iloc[val_index] 
    y_val_fold= y_dev_encoded[val_index]

    # ----------XGBoost model------------
    model_xgb= XGBClassifier(n_estimators=700, max_depth=6, learning_rate=0.03,subsample=0.8 ,tree_method='hist',
    colsample_bytree=0.8,num_class=3,random_state=42+fold,enable_categorical=True,objective="multi:softprob") 

    model_xgb.fit(X_train_fold,y_train_fold)

    #Prediction et accuracy du fold
    modele_xgb_pred= model_xgb.predict(X_val_fold)
    modele_xgb_accuracy=accuracy_score(y_val_fold,modele_xgb_pred)
    scores_xgb.append(modele_xgb_accuracy)

    #Stockage des probabilités OOF
    proba_xgb_fold=model_xgb.predict_proba(X_val_fold)
    oof_xgb[val_index]=proba_xgb_fold

    #Calculer la moyenne des proba predites sur test par les 5 modèles XGBoost
    test_probs_xgb+=model_xgb.predict_proba(X_test_model)/ skf.n_splits



    #-------------LightGBM model---------------
    model_Lgbm =LGBMClassifier(n_estimators=700, learning_rate=0.03,num_leaves=31,max_depth=8,verbosity=-1, random_state=42+fold,
    subsample=0.8,colsample_bytree=0.8,objective="multiclass",num_class=3)    
    model_Lgbm.fit(X_train_fold,y_train_fold)

    #Prediction et accuracy du fold
    modele_Lgbm_pred=model_Lgbm.predict(X_val_fold)
    modele_Lgbm_accuracy=accuracy_score(y_val_fold,modele_Lgbm_pred)
    scores_lgbm.append(modele_Lgbm_accuracy)

    #Stockage des probabilités OOF
    proba_lgbm_fold=model_Lgbm.predict_proba(X_val_fold)
    oof_lgbm[val_index]=proba_lgbm_fold

    #La moyenne des proba predites sur test par les 5 modèles de LightGBM 
    test_probs_lgbm+=model_Lgbm.predict_proba(X_test_model)/ skf.n_splits

# L'accuracy OOF global de XGBoost et de LightGBM    
y_pred_oof_pred=np.argmax(oof_xgb,axis=1)
accuracy_oof_xgb=accuracy_score(y_dev_encoded,y_pred_oof_pred)

y_pred_oof_lgbm=np.argmax(oof_lgbm,axis=1)
accuracy_oof_Lgbm=accuracy_score(y_dev_encoded,y_pred_oof_lgbm)



print("-"*60)
print("Evaluation des modeles-Validation Croisée(5 Folds)")
print("-"*60)

print("\n ----------XGboost----------")
print("Accuracy moyenne :",np.mean(scores_xgb))    
print("Ecart-type :",np.std(scores_xgb))  
print("Accuracy OOF",accuracy_oof_xgb)

print("\n ----------LightGBM----------")

print("Accuracy moyenne :",np.mean(scores_lgbm))    
print("Ecart-type :",np.std(scores_lgbm))  
print("Accuracy OOF",accuracy_oof_Lgbm)      

----Fold 1---
----Fold 2---
----Fold 3---
----Fold 4---
----Fold 5---
------------------------------------------------------------
Evaluation des modeles-Validation Croisée(5 Folds)
------------------------------------------------------------

 ----------XGboost----------
Accuracy moyenne : 0.8323139458696656
Ecart-type : 0.0012099851564513317
Accuracy OOF 0.8323139653414883

 ----------LightGBM----------
Accuracy moyenne : 0.8323923661364363
Ecart-type : 0.0009001169495809677
Accuracy OOF 0.8323923782639379


In [31]:
#curacy moyenne : 0.8326929548132089
#uracy moyenne : 0.8327844316030678
#Validation croisée sur 5 plis pour évaluer les performances du  modèle CatBoost
scores_cb=[]
oof_catboost=np.zeros((len(X_dev_cb), len(encoder.classes_)))
test_probs_cb=np.zeros((len(X_test_cb), len(encoder.classes_)))

for fold,(train_index, val_index) in enumerate(skf.split(X_dev_cb,y_dev_encoded)):
    print(f"----Fold {fold+1}---")
    X_train_fold= X_dev_cb.iloc[train_index]
    y_train_fold= y_dev_encoded[train_index]
    X_val_fold= X_dev_cb.iloc[val_index] 
    y_val_fold= y_dev_encoded[val_index] 

    #-------------CatBoost model------------------
    model_catboost =CatBoostClassifier(iterations=700, learning_rate=0.03, depth=6,verbose=False, random_state=42+fold,
    loss_function='MultiClass',cat_features=cat_cols)

    model_catboost.fit(X_train_fold,y_train_fold) 

    #Prediction  et accuracy du fold
    modele_catboost_pred=model_catboost.predict(X_val_fold).ravel()
    modele_catboost_accuracy=accuracy_score(y_val_fold,modele_catboost_pred)
    scores_cb.append(modele_catboost_accuracy)

    #Stockage des probabiltés 
    proba_catboost_fold=model_catboost.predict_proba(X_val_fold)
    oof_catboost[val_index]=proba_catboost_fold

    #Calculer la moyenne des proba predites sur test par les 5 modèles CatBoost 
    test_probs_cb+=model_catboost.predict_proba(X_test_cb)/skf.n_splits

#Accuracy OOF Global
y_pred_oof_catboost=np.argmax(oof_catboost,axis=1)
accuracy_oof_catboost=accuracy_score(y_dev_encoded,y_pred_oof_catboost)

print("\n ----------CatBoost----------")

print("Accuracy moyenne :",np.mean(scores_cb))    
print("Ecart-type :",np.std(scores_cb))  
print("Accuracy OOF",accuracy_oof_catboost)


----Fold 1---
----Fold 2---
----Fold 3---
----Fold 4---
----Fold 5---

 ----------CatBoost----------
Accuracy moyenne : 0.8290597665440043
Ecart-type : 0.002257741152023572
Accuracy OOF 0.8290598290598291


In [32]:
comparaison_modeles=pd.DataFrame({
    "Modele":["XGBoost", "LightGBM", "CatBoost"],
    "Accuracy OOF": [accuracy_oof_xgb,accuracy_oof_Lgbm,accuracy_oof_catboost]
})
display(comparaison_modeles.style.format({"Accuracy OOF": "{:.4%}"}))

,Modele,Accuracy OOF
0,XGBoost,83.2314%
1,LightGBM,83.2967%
2,CatBoost,82.9060%


In [98]:
# Recherche des meilleurs poids du blending
best_accuracy = 0.0
best_weights = (0, 0, 0)

for w_cb in np.linspace(0, 1,21):
    for w_lgbm in np.linspace(0, 1 - w_cb,21):

        w_xgb = 1.0 - w_cb - w_lgbm

        # Vérifier que les poids sont valides
        if w_xgb < -1e-5:
            continue

        # Combinaison pondérée des probabilités OOF
        proba_blend = (w_cb * oof_catboost+ w_lgbm * oof_lgbm + w_xgb * oof_xgb)

        # Accuracy OOF
        y_pred = np.argmax(proba_blend, axis=1)
        score = accuracy_score(y_dev_encoded, y_pred)

        # Conserver les meilleurs poids
        if score > best_accuracy:
            best_accuracy = score
            best_weights = (w_xgb, w_lgbm, w_cb)

print(f"La meilleure accuracy OOF est : {best_accuracy:.5%}")
print(f"Les meilleurs poids sont : {tuple(round(w, 4) for w in best_weights)}")





La meilleure accuracy OOF est : 83.31373%
Les meilleurs poids sont : (0.665, 0.285, 0.05)


In [99]:
# La prediction finales sur le blending optimisé

w_xgb_final,w_lgbm_final,w_cb_final=best_weights

#combinaison pondéré des probabilités 
proba_test_blend=(w_xgb_final * test_probs_xgb  + w_lgbm_final * test_probs_lgbm +w_cb_final * test_probs_cb )

# Sélectionner la classe ayant la probabilité la plus élevée
y_pred_test = np.argmax(proba_test_blend, axis=1)

# Convertir les classes numériques en labels
predictions_finales = encoder.inverse_transform(y_pred_test)

print("Dimensions des probabilités :", proba_test_blend.shape)
print("Nombre de prédictions :", len(predictions_finales))
print("Classes prédites :", np.unique(predictions_finales))

Dimensions des probabilités : (51012, 3)
Nombre de prédictions : 51012
Classes prédites : ['Dropout' 'Enrolled' 'Graduate']


In [100]:
# Créer le fichier de soumission Kaggle
submission = pd.DataFrame({"id": test["id"],"Target": predictions_finales})

# Enregistrer le fichier CSV
submission.to_csv("submission_Bestmodel.csv", index=False)
print("Fichier submission_Bestmodel.csv créé avec succès !")
print("Dimensions :", submission.shape)
display(submission.head())


Fichier submission_Bestmodel.csv créé avec succès !
Dimensions : (51012, 2)


,id,Target
0,76518,Dropout
1,76519,Graduate
2,76520,Graduate
3,76521,Enrolled
4,76522,Enrolled
